# Lezione 8. Gradienti di politica e metodi Actor-Critic

**DIH DATalife — Corso di Reinforcement Learning (RL) da zero**

Benvenuta o benvenuto alla lezione otto. Finora il tuo agente imparava quanto vale ogni azione e, poi, agiva scegliendo sempre quella di valore più alto. Oggi cambi approccio: impari la politica direttamente, senza quel passaggio intermedio. Capirai perché questo conta, quando agire a caso è la strategia migliore possibile, e programmerai REINFORCE, l'algoritmo di gradiente di politica più semplice, su CartPole.

## Scaletta della lezione

1. Dalla funzione di valore alla politica diretta
2. Perché a volte conviene agire a caso
3. Il trucco della funzione score
4. REINFORCE, passo dopo passo
5. Ridurre la varianza: baseline e vantaggio
6. Architettura Actor-Critic
7. Codice dal vivo: REINFORCE su CartPole
8. Punti chiave e cosa ti aspetta nella lezione 9


## 1. Dalla funzione di valore alla politica diretta

*Traccia della lezione: ricorda ad alta voce come funzionava Deep Q-Network (DQN) nella lezione sette, e chiedi alla classe cosa succederebbe se il numero di azioni possibili fosse infinito.*

Deep Q-Network (DQN), che hai visto nella lezione precedente, impara quanto vale ogni azione in ogni stato tramite una rete neurale, e poi agisce scegliendo sempre l'azione di valore più alto. Funziona molto bene, ma fa un giro largo: prima impara una funzione di valore completa e solo dopo, in modo indiretto, deriva una politica da essa.

Oggi vedi un'idea diversa: invece di imparare quanto vale ogni azione, impari direttamente la politica, la regola che decide cosa fare in ogni stato. Pensa a un robot la cui strategia migliore è semplicemente "avanzare sempre": la sua politica ottimale è banale, ma la funzione di valore che descrive quanto vale ogni combinazione di posizione e ostacoli può essere enormemente complessa. Se il tuo obiettivo reale è agire bene, perché imparare prima quel giro complicato invece di andare dritto alla politica? Questo principio —non risolvere un problema più generale di quello che ti serve per risolvere il problema concreto che ti interessa— è attribuito allo statistico Vladimir Vapnik, ed è la motivazione centrale della lezione di oggi.

Imparare la politica direttamente ha, inoltre, due vantaggi pratici che DQN non offre con la stessa naturalezza:

- **Spazi di azione continui o molto grandi**: pensa all'angolo di un motore o alla forza di un braccio robotico. DQN deve calcolare il massimo su tutte le azioni possibili, cosa impraticabile quando sono infinite. Un metodo basato sulla politica, invece, emette semplicemente l'azione (o una distribuzione sulle azioni) senza bisogno di quel massimo
- **Politiche genuinamente stocastiche**, non solo deterministiche, cosa che capirai nel prossimo paragrafo


## 2. Perché a volte conviene agire a caso

*Traccia della lezione: presenta il labirinto come domanda aperta prima di dare la risposta. "Cosa succede a un agente deterministico, qui?"*

Immagina un labirinto con due corridoi che, visti da dentro, sembrano esattamente uguali: stesse pareti a sinistra e a destra, stesso aspetto. Un agente che sa agire solo in modo deterministico può restare intrappolato, passando da un corridoio all'altro in un ciclo infinito, perché dal suo punto di vista i due stati sono indistinguibili. Una politica stocastica, che in quei corridoi sceglie a caso sinistra o destra, prima o poi esce dal ciclo per pura probabilità e arriva in una zona dove può distinguere dove si trova.

Lo stesso accade nei giochi con informazione nascosta. Nella morra cinese, qualsiasi strategia deterministica è sfruttabile immediatamente: il tuo avversario deve solo individuare lo schema. L'unica strategia ottimale è scegliere ogni opzione con la stessa probabilità. Nel poker succede qualcosa di simile: se il tuo modo di agire con una mano vincente è sempre lo stesso, un avversario attento lo impara e lo sfrutta; il bluff funziona solo se è probabilistico.

Una politica, quindi, non è più una regola fissa "stato → azione" (una funzione deterministica `a = μ(s)`), ma una distribuzione di probabilità sulle azioni dato lo stato:

`π(a|s)`

Imparare questa distribuzione, invece di una regola fissa, è ciò che rende possibile rappresentare comportamenti come quelli del labirinto o del poker.


## 3. Il trucco della funzione score

*Traccia della lezione: non perderti nello sviluppo matematico completo. Trasmetti solo l'idea intuitiva e la formula finale, sottolineando la praticità della conseguenza (non serve modellare l'ambiente).*

Qui sorge un problema tecnico. Vuoi regolare i parametri della tua politica, θ, per massimizzare la ricompensa attesa. Ma quella ricompensa attesa dipende da un processo di campionamento casuale: l'azione che prendi viene estratta da una distribuzione di probabilità, e non puoi "derivare" direttamente attraverso un sorteggio.

La soluzione è nota come trucco della funzione score (o trucco REINFORCE, o trucco della log-derivata). L'idea intuitiva, senza lo sviluppo matematico completo: invece di cercare di derivare il risultato di un campionamento, trasformi il problema in qualcosa che sai derivare, il logaritmo della probabilità dell'azione che hai scelto. Il risultato è sorprendentemente semplice da applicare: per regolare la politica, calcoli il gradiente del logaritmo della probabilità di ogni azione che hai preso, e lo moltiplichi per quanto è stato buono quel risultato (la sua ricompensa o il suo ritorno).

Detto in parole semplici: alza la probabilità delle azioni che hanno portato a una ricompensa alta, e abbassa la probabilità di quelle che hanno portato a una ricompensa bassa o negativa. Quanto migliore è stato il risultato, tanto più forte è la spinta verso l'alto.

**Un dettaglio pratico importante**: quando scomponi la probabilità di una traiettoria completa, compaiono due tipi di termini: quelli che dipendono dalla dinamica dell'ambiente (quale stato segue quale) e quelli che dipendono solo dalla tua politica. Quando calcoli il gradiente rispetto ai parametri della politica, i termini della dinamica dell'ambiente scompaiono, perché non dipendono da θ. Questo significa che i metodi di gradiente di politica non hanno bisogno di conoscere né modellare il funzionamento dell'ambiente: basta poter agire in esso e osservare cosa succede.


## 4. REINFORCE: l'algoritmo più semplice

*Traccia della lezione: scrivi i quattro passi alla lavagna prima di mostrare il codice. Insisti sul fatto che "ritorno" qui è la somma delle ricompense future dell'episodio completo, non un valore stimato da una rete.*

Con questa idea hai già l'algoritmo REINFORCE, la versione più elementare del gradiente di politica:

1. Inizializza i parametri θ della politica, di solito una rete neurale
2. Ripeti finché le prestazioni ti convincono:
   - Esegui un episodio completo seguendo la politica attuale e salva stati, azioni e ricompense
   - Per ogni passo dell'episodio, calcola il ritorno (la somma delle ricompense future, di solito scontate)
   - Calcola il gradiente: per ogni azione presa, moltiplica il gradiente del logaritmo della sua probabilità per il ritorno corrispondente
   - Aggiorna i parametri nella direzione di quel gradiente: `θ ← θ + α · gradiente`

È un algoritmo elegante e facile da programmare. Ma ha un problema serio: alta varianza. Ogni episodio dipende da decisioni casuali (quali azioni ha estratto il campionamento) e dalla casualità propria dell'ambiente, quindi il ritorno di un episodio può essere molto diverso da quello di un altro episodio quasi identico. Questo fa sì che gli aggiornamenti saltino da una parte all'altra e che l'addestramento sia lento e instabile. Lo vedrai con i tuoi occhi tra poco, nel grafico del codice dal vivo.


## 5. Ridurre la varianza: baseline e vantaggio

*Traccia della lezione: chiedi, prima di spiegare perché sottrarre un numero costante non cambia la direzione del gradiente. Lascia che qualcuno della classe provi a ragionarci.*

Qui entra un'idea molto elegante. Puoi sottrarre dal ritorno una baseline, `b(s)`, un valore che dipende solo dallo stato (non dall'azione scelta), senza introdurre alcun bias nell'apprendimento. Il motivo è che, sommando su tutte le azioni possibili, l'effetto di una baseline costante si annulla matematicamente: non cambia la direzione del gradiente, riduce solo la sua varianza.

La baseline più comune è la stessa funzione di valore di stato, `V(s)`: quanta ricompensa ti aspetti di ottenere da quello stato seguendo la tua politica attuale. Sottraendola dal valore di azione `Q(s,a)`, ottieni la funzione di vantaggio (*advantage*):

`A(s,a) = Q(s,a) − V(s)`

Il vantaggio risponde a una domanda molto precisa: quanto è migliore (o peggiore) questa azione concreta rispetto a ciò che faresti in media in questo stato? Un vantaggio positivo spinge verso l'alto la probabilità di quell'azione; un vantaggio negativo la spinge verso il basso. È un segnale molto più pulito del ritorno grezzo, perché non trascina più la parte della ricompensa che dipende solo dallo stato in cui ti trovavi, ma solo la parte che dipende dalla decisione che hai preso.

Nella pratica non sempre servono due reti separate per Q e V. Puoi stimare il vantaggio con l'errore di differenza temporale (TD, dalla sigla di "temporal difference"): `δ = r + γV(s') − V(s)`. Questo errore confronta ciò che ti aspettavi di ottenere con ciò che hai ottenuto davvero dopo un passo, ed è, in pratica, un campione del vantaggio.


## 6. Architettura Actor-Critic

*Traccia della lezione: racconta l'analogia dello chef e dell'esperto di gastronomia prima di mostrare lo pseudocodice. Collega ogni riga dello pseudocodice al suo ruolo nell'analogia.*

Quest'ultima idea porta direttamente ai metodi Actor-Critic. Immagina uno chef (l'attore) che decide quale piatto cucinare, e un esperto di gastronomia (il critico) che assaggia ogni piatto e gli dice quanto è stato meglio o peggio del previsto. Lo chef usa quella valutazione per regolare le sue prossime decisioni.

In termini tecnici, l'attore è la politica `π_θ`, che decide quale azione prendere, e il critico è una funzione di valore `V_ω`, che valuta quanto bene sta andando la prestazione dell'attore. Il critico si addestra con tecniche di differenza temporale, come la funzione di valore di DQN. L'attore si addestra con il gradiente di politica, usando l'errore TD del critico come segnale di vantaggio. Lo schema di base è questo:

```
Inizializza la rete dell'attore (politica) con parametri θ
Inizializza la rete del critico (valore) con parametri ω

per ogni episodio:
    inizializza lo stato s
    per ogni passo dell'episodio:
        campiona un'azione a ~ π_θ(a|s)
        esegui a, osserva la ricompensa r e lo stato successivo s'
        δ = r + γ · V_ω(s') − V_ω(s)          # errore TD, usato come vantaggio
        ω ← ω + α_critico · δ · ∇_ω V_ω(s)     # aggiorna il critico
        θ ← θ + α_attore · δ · ∇_θ log π_θ(a|s) # aggiorna l'attore
        s ← s'
```

Questo schema, in cui il critico impara a ogni passo (non serve aspettare la fine dell'episodio) e l'attore usa quel segnale per migliorare, si chiama Advantage Actor-Critic (A2C), e la sua variante che esegue più agenti in parallelo, ciascuno con una propria copia dell'ambiente, si chiama Asynchronous Advantage Actor-Critic (A3C). Ritroverai A3C nel capitolo 12 del manuale.

Esistono evoluzioni successive di queste idee che troverai citate più avanti nel corso, senza entrare oggi nei dettagli tecnici: Trust Region Policy Optimization (TRPO) e Proximal Policy Optimization (PPO), che limitano quanto può cambiare la politica a ogni aggiornamento per evitare passi troppo bruschi, e Deep Deterministic Policy Gradient (DDPG), pensato per spazi di azione continui.


## 7. Codice dal vivo: REINFORCE su CartPole

*Traccia della lezione: costruisci il codice cella per cella, senza copiare e incollare l'intero blocco. Prima di eseguire la cella del ciclo di addestramento, chiedi alla classe cosa si aspetta di vedere nel grafico finale.*

Implementerai REINFORCE da zero su CartPole, l'ambiente classico in cui un carrello deve mantenere in equilibrio un palo spostandosi a sinistra o a destra. Tutto il codice funziona in Google Colab: usa PyTorch (già preinstallato) e gymnasium (si installa nella prima cella). Non serve una GPU.

**Esegui ogni cella in ordine, dall'alto in basso.**


### Passo 1 — Installa gymnasium

PyTorch è già preinstallato in Colab, quindi ti manca solo gymnasium, la libreria degli ambienti.


In [ ]:
# Celda 1: instala gymnasium (PyTorch ya viene preinstalado en Colab)
!pip install gymnasium

### Passo 2 — Importa le librerie e fissa il seme

*Traccia della lezione: ricorda perché fissiamo i semi — riproducibilità, non casualità reale.*

Fissi il seme di PyTorch e di NumPy perché l'esperimento sia riproducibile: con lo stesso seme ottieni sempre la stessa sequenza di numeri casuali.


In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

# Fijamos semillas para que el experimento sea reproducible
SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

### Passo 3 — Definisci la rete della politica

`RedPolitica` rappresenta `π_θ(a|s)`: riceve lo stato di CartPole (quattro numeri —posizione e velocità del carrello, angolo e velocità angolare del palo—) e restituisce una distribuzione di probabilità sulle due azioni possibili (spingere il carrello a sinistra o a destra). Lo strato `softmax` finale converte i `logits` della rete in probabilità che sommano a uno.


In [ ]:
class RedPolitica(nn.Module):
    """Red neuronal pequeña que representa la política π_θ(a|s).
    Recibe el estado de CartPole (cuatro números: posición y velocidad del
    carro, ángulo y velocidad angular del palo) y devuelve una distribución
    de probabilidad sobre las dos acciones posibles (empujar el carro a la
    izquierda o a la derecha)."""

    def __init__(self, n_entradas, n_acciones, n_ocultas=128):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(n_entradas, n_ocultas),
            nn.ReLU(),
            nn.Linear(n_ocultas, n_acciones),
        )

    def forward(self, estado):
        logits = self.red(estado)
        return torch.softmax(logits, dim=-1)

### Passo 4 — Calcola i ritorni scontati

`calcular_retornos` implementa il "reward-to-go": a ogni passo `t`, somma le ricompense future dell'episodio, ciascuna scontata di `gamma` elevato alla distanza in passi. Scorrere la lista delle ricompense dalla fine all'inizio ti permette di riutilizzare il calcolo del passo successivo a ogni passo precedente, invece di ricalcolare la somma completa ogni volta.


In [ ]:
def calcular_retornos(recompensas, gamma=0.99):
    """Calcula el retorno 'reward-to-go' en cada paso del episodio:
    G_t = r_t + gamma*r_{t+1} + gamma^2*r_{t+2} + ...
    Se recorre la lista de atrás hacia adelante para reutilizar el cálculo
    del paso siguiente en cada paso anterior."""
    retornos = np.zeros(len(recompensas))
    G = 0.0
    for t in reversed(range(len(recompensas))):
        G = recompensas[t] + gamma * G
        retornos[t] = G
    return retornos

### Passo 5 — Raccogli un episodio completo

`ejecutar_episodio` segue la politica attuale dal reset dell'ambiente fino alla fine dell'episodio. A ogni passo campiona un'azione dalla distribuzione `π_θ(a|s)` (con `torch.distributions.Categorical`) e salva la sua log-probabilità, che ti serve dopo per il gradiente di politica.


In [ ]:
def ejecutar_episodio(env, politica):
    """Recolecta un episodio completo siguiendo la política actual.
    Devuelve la log-probabilidad de cada acción tomada (necesaria para el
    gradiente de política) y la recompensa obtenida en cada paso."""
    log_probs = []
    recompensas = []
    estado, _ = env.reset()
    terminado = False
    truncado = False

    while not (terminado or truncado):
        estado_tensor = torch.FloatTensor(estado).unsqueeze(0)
        probs = politica(estado_tensor)
        distribucion = torch.distributions.Categorical(probs)
        accion = distribucion.sample()

        log_probs.append(distribucion.log_prob(accion))
        estado, recompensa, terminado, truncado, _ = env.step(accion.item())
        recompensas.append(recompensa)

    return log_probs, recompensas

### Passo 6 — Configura l'ambiente e gli iperparametri

Crei l'ambiente CartPole-v1, istanzi la rete della politica e l'ottimizzatore Adam, e fissi i due iperparametri centrali: il numero di episodi di addestramento e `GAMMA`, il fattore di sconto.


In [ ]:
env = gym.make("CartPole-v1")
n_entradas = env.observation_space.shape[0]
n_acciones = env.action_space.n

politica = RedPolitica(n_entradas, n_acciones)
optimizador = optim.Adam(politica.parameters(), lr=1e-2)

N_EPISODIOS = 500
GAMMA = 0.99

historial_recompensas = []

### Passo 7 — Il ciclo di addestramento

*Traccia della lezione: fermati sulla riga della perdita. Chiedi: "perché c'è un segno negativo?" prima di leggere ad alta voce il commento.*

Questo è il cuore di REINFORCE. Per ogni episodio:

1. Raccogli l'esperienza con `ejecutar_episodio`
2. Calcoli il ritorno di ogni passo con `calcular_retornos`
3. Costruisci la perdita: `-sum(log_prob(a_t|s_t) * G_t)`. Il segno negativo trasforma la salita del gradiente che vuoi (massimizzare la ricompensa attesa) nella discesa del gradiente che PyTorch ottimizza di default
4. Esegui la retropropagazione e aggiorni i parametri della politica con l'ottimizzatore

**Esegui questa cella e osserva come sale la ricompensa media ogni 50 episodi.** In CartPole-v1, un episodio dura al massimo 500 passi, quindi una ricompensa media vicina a quel numero indica che il palo resta in equilibrio quasi tutto il tempo consentito.


In [ ]:
for episodio in range(N_EPISODIOS):
    log_probs, recompensas = ejecutar_episodio(env, politica)
    retornos = calcular_retornos(recompensas, GAMMA)
    retornos_tensor = torch.FloatTensor(retornos)

    # Pérdida del gradiente de política: -sum(log_prob(a_t|s_t) * G_t)
    # El signo negativo convierte el ASCENSO de gradiente que queremos
    # (maximizar la recompensa esperada) en el DESCENSO de gradiente que
    # optimiza PyTorch por defecto.
    perdida = torch.stack(
        [-log_prob * G for log_prob, G in zip(log_probs, retornos_tensor)]
    ).sum()

    optimizador.zero_grad()
    perdida.backward()
    optimizador.step()

    historial_recompensas.append(sum(recompensas))

    if (episodio + 1) % 50 == 0:
        media_reciente = np.mean(historial_recompensas[-50:])
        print(f"Episodio {episodio + 1}: recompensa media (últimos 50) = {media_reciente:.1f}")

env.close()

### Passo 8 — Grafica la ricompensa per episodio

La curva sottile mostra la ricompensa episodio per episodio; la curva spessa è una media mobile che attenua il rumore per far vedere meglio la tendenza.

**Esegui questa cella e osserva i saliscendi tra episodi vicini.**


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(historial_recompensas, alpha=0.4, label="Recompensa por episodio")

ventana = 20  # media móvil para ver la tendencia sin el ruido episodio a episodio
media_movil = np.convolve(historial_recompensas, np.ones(ventana) / ventana, mode="valid")
plt.plot(range(ventana - 1, len(historial_recompensas)), media_movil, label=f"Media móvil ({ventana} episodios)")

plt.xlabel("Episodio")
plt.ylabel("Recompensa total")
plt.title("REINFORCE en CartPole-v1")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### Lettura del risultato

*Traccia della lezione: fermati qui e chiedi alla classe quale curva si aspettavano rispetto a quella che vedono. Collega esplicitamente questo al problema della varianza del paragrafo quattro.*

Eseguendolo vedi qualcosa di tipico di REINFORCE: la curva della ricompensa sale, ma con molti saliscendi tra episodi vicini, esattamente il problema di varianza che conosci dal paragrafo quattro. Questo andirivieni è la ragione d'essere della baseline e del vantaggio del paragrafo cinque: nel notebook di pratica di questa lezione verificherai di persona quanto aiutano a rendere più liscia questa curva.


## Punti chiave

- I metodi di gradiente di politica imparano direttamente la politica, senza passare per una funzione di valore completa, il che si adatta meglio a spazi di azione continui o molto grandi
- Una politica stocastica, `π(a|s)`, è necessaria negli stati ambigui e nei giochi con informazione nascosta, dove una regola deterministica sarebbe sfruttabile
- Il trucco della funzione score trasforma un problema di derivazione attraverso un campionamento casuale in qualcosa di calcolabile: il gradiente del logaritmo della probabilità dell'azione, moltiplicato per quanto è stato buono il risultato
- REINFORCE applica questa idea in modo diretto, ma soffre di alta varianza perché dipende da ritorni completi di episodi rumorosi
- Sottrarre una baseline (di solito `V(s)`) riduce la varianza senza introdurre bias, e dà origine alla funzione di vantaggio `A(s,a) = Q(s,a) − V(s)`
- L'architettura Actor-Critic combina un attore (la politica) con un critico (una funzione di valore) che valuta ogni passo e guida l'aggiornamento dell'attore con l'errore TD
- A2C, A3C, PPO, TRPO e DDPG sono evoluzioni di queste idee che incontrerai nelle prossime lezioni, senza bisogno di padroneggiarne ancora il dettaglio tecnico


## Prima della prossima lezione

**Apri il notebook di pratica di questa lezione e completa i due esercizi su REINFORCE.** Programmerai tu stesso il calcolo dei ritorni scontati e la perdita di REINFORCE da zero, e aggiungerai una baseline per verificare in prima persona quanto riduce la varianza dell'addestramento.

Nella lezione 9 cambi famiglia di metodi: vedrai cosa succede quando l'agente costruisce o usa un modello dell'ambiente per pianificare in anticipo, con tecniche come Dyna-Q e la ricerca ad albero Monte Carlo (Monte Carlo Tree Search).
